# Proportional control of a first-order plant

A first-order plant with the time constant $\tau$ and the static gain $K$,

$$\tau\,\frac{dy}{dt}=-y+K\,u,$$

is driven by a proportional controller $u=K_p\,(r-y)$ that compares the output
$y$ with a constant setpoint $r$. OneSimulation builds the loop from signal
blocks (setpoint, summing junction, controller, plant) and integrates the single
continuous state with the classical fourth-order Runge-Kutta method.

**Reference.** Substituting the control law gives another first-order system,
so the closed-loop step response from rest is

$$y(t)=y_\infty\left(1-e^{-t/\tau_\mathrm{cl}}\right),\qquad
  y_\infty=\frac{K K_p}{1+K K_p}\,r,\qquad
  \tau_\mathrm{cl}=\frac{\tau}{1+K K_p}.$$

A proportional controller speeds the plant up by the factor $1+KK_p$ but leaves
the steady-state offset $r-y_\infty=r/(1+KK_p)$. Both facts are checked below.

In [ ]:
from importlib.metadata import version
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from cycler import cycler
from matplotlib.colors import LinearSegmentedColormap

# OneSimulation figure style: cream paper, dark-green ink, green-sand-copper field colours
CREAM, INK, GREEN, COPPER = "#F7F2E7", "#223D35", "#336B57", "#B66E43"
FIELD_CMAP = LinearSegmentedColormap.from_list(
    "onesimulation", ["#214D43", "#5F9175", "#A8BC91", "#E4D6A0", "#CD9561", "#A85539"])
DIVERGING_CMAP = LinearSegmentedColormap.from_list(
    "onesimulation_diverging", ["#104F54", "#5E9996", "#F7F2E7", "#D2996E", "#A4472D"])
for _cmap in (FIELD_CMAP, DIVERGING_CMAP):
    if _cmap.name not in matplotlib.colormaps:
        matplotlib.colormaps.register(_cmap)
plt.rcParams.update({
    "figure.facecolor": CREAM, "savefig.facecolor": CREAM, "axes.facecolor": "#FFFAF2",
    "text.color": INK, "axes.labelcolor": INK, "axes.titlecolor": INK, "axes.edgecolor": "#9FA593",
    "xtick.color": INK, "ytick.color": INK, "grid.color": "#D5D7C8", "image.cmap": "onesimulation",
    "axes.prop_cycle": cycler(color=[GREEN, COPPER, "#5F9175", "#A85539", "#214D43", "#CD9561", "#8B9A8D"]),
})

from onesim.physics.systems import ControlSignalFlowProvider, SignalBlock

print("OneSimulation:", version("onesimulation"))

## Parameters and model

The parameters can be changed; the reference stays valid for any positive gains
and time constant. The values describe a sluggish process (20 s time constant,
gain 2) under a moderate proportional gain, so the offset remains clearly visible.

In [ ]:
time_constant = 20.0        # s, plant time constant tau
plant_gain = 2.0            # static plant gain K
proportional_gain = 4.0     # controller gain K_p
setpoint = 1.0              # r, unit step

loop_gain = plant_gain * proportional_gain
steady_output = loop_gain / (1.0 + loop_gain) * setpoint
closed_loop_time_constant = time_constant / (1.0 + loop_gain)
end_time = 6.0 * closed_loop_time_constant
time_step = 0.01            # s
print(f"loop gain K*Kp = {loop_gain:.1f},  closed-loop time constant = {closed_loop_time_constant:.3f} s "
      f"(open loop {time_constant:.1f} s),  steady output = {steady_output:.4f}")

loop = ControlSignalFlowProvider(
    blocks=(
        SignalBlock("setpoint", "constant", parameters={"value": setpoint}),
        SignalBlock("error", "sum", inputs=("setpoint", "plant"), parameters={"weights": (1.0, -1.0)}),
        SignalBlock("controller", "pid", inputs=("error",), parameters={"proportional_gain": proportional_gain}),
        SignalBlock("actuator", "gain", inputs=("controller",), parameters={"gain": plant_gain}),
        SignalBlock("plant", "first_order", inputs=("actuator",),
                    parameters={"time_constant": time_constant, "initial_value": 0.0}),
    ),
    final_time=end_time,
    time_step=time_step,
    time_integration="classical_rk4",
)
run = loop.execute()
times = np.asarray(run.times)
outputs = np.asarray(run.signals["plant"])
commands = np.asarray(run.signals["controller"])
print("evaluation order:", " -> ".join(run.evaluation_order))

## Comparison with the closed-form solution

The closed-loop exponential is evaluated at every stored time, the closed-loop
time constant is fitted to the simulated approach, and the control law is
verified sample by sample. A passing check shows that this numerical example
agrees with the linear loop equations; it is not a measurement of a real process.

In [ ]:
reference = steady_output * (1.0 - np.exp(-times / closed_loop_time_constant))
max_output_error = float(np.max(np.abs(outputs - reference)))
print(f"Maximum output error against the closed-loop exponential: {max_output_error:.3e}")


# Recover the closed-loop time constant from the log-slope of the remaining approach.
approach = steady_output - outputs
fitted_rate = -np.polyfit(times, np.log(approach), 1)[0]
fitted_time_constant = 1.0 / fitted_rate
print(f"Closed-loop time constant: fitted {fitted_time_constant:.5f} s, exact {closed_loop_time_constant:.5f} s")
final_offset = setpoint - outputs[-1]
expected_offset = setpoint / (1.0 + loop_gain)
print(f"Steady-state offset: simulated {final_offset:.5f}, exact r/(1+K Kp) = {expected_offset:.5f}")

## Figure

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True, constrained_layout=True)
fig.patch.set_facecolor("#fbf5eb")
for ax in axes:
    ax.set_facecolor("#fffaf2")
    ax.grid(alpha=0.25)
dense_t = np.linspace(0.0, end_time, 400)
dense_y = steady_output * (1.0 - np.exp(-dense_t / closed_loop_time_constant))
stride = max(1, len(times) // 40)
axes[0].plot(dense_t, dense_y, color="#6b442f", lw=1.6, label=r"$y_\infty\,(1-e^{-t/\tau_{cl}})$")
axes[0].plot(times[::stride], outputs[::stride], "o", color="#c56838", ms=4, label="OneSimulation RK4")
axes[0].axhline(setpoint, color="#39715E", ls="--", lw=1.0, label="Setpoint $r$")
axes[0].axhline(steady_output, color="#9a8f7d", ls="--", lw=1.0, label=r"$y_\infty=KK_p r/(1+KK_p)$")
axes[0].axvline(closed_loop_time_constant, color="#9a8f7d", ls=":", lw=1.0, label=r"$t=\tau_{cl}$")
axes[0].set(ylabel="Plant output $y$", title="First-order plant under proportional control")
axes[0].legend(loc="lower right", fontsize=9)
axes[1].plot(times, commands, color="#c56838", lw=1.4, label=r"Controller output $u=K_p(r-y)$")
axes[1].axhline(proportional_gain * expected_offset, color="#9a8f7d", ls="--", lw=1.0,
                label=r"$u_\infty=K_p r/(1+KK_p)$")
axes[1].set(xlabel="Time / s", ylabel="Control signal $u$")
axes[1].legend(loc="upper right")
plt.show()

## Going further

Raise `proportional_gain` and the offset shrinks as $1/(1+KK_p)$ while the loop
becomes faster; it can never vanish with proportional action alone. Add an
`integral_gain` to the controller block and the offset disappears, but the loop
becomes second order and may overshoot; derive the new characteristic equation
$\tau s^2+(1+KK_p)s+KK_i=0$ before adapting the checks. A `saturation` block
between controller and actuator or a `transport_delay` in the feedback path
turns the example into the standard anti-windup and dead-time studies.